# Raster workflow

Index daily OpenET GeoTIFFs and footprints, extract model ET with footprint and fixed-window
supports, and describe the landscape each support sees. Needs the `geo` extra. The demonstration
rasters are **synthetic** (fixed seed); point `CONFIG` at your own configuration to use yours.

In [ ]:
# Install etcompare (Colab or any fresh environment). Skip this cell if it is installed.
# From a checkout or wheel you have, e.g.:  %pip install "/content/comparison[geo,plot]"
%pip install --quiet "etcompare[geo,plot] @ git+https://github.com/inkenbrandt/comparison"

In [ ]:
from pathlib import Path

WORKDIR = Path("etcompare_rasters")
CONFIG = None                      # your config.yaml with openet_rasters / raster_extractions

In [ ]:
import pandas as pd
from etcompare.config import load_config
from etcompare.demo import write_demo
from etcompare.runner import run_stages

cfg = load_config(CONFIG or write_demo(WORKDIR, rasters=True))
print("raster root:", cfg.openet_rasters[0].root)
print("extractions:", [x.name for x in cfg.raster_extractions])

## Index and extract

Indexing parses filenames only; extraction reads each raster once, in a bounded window, and moves
the footprint onto the model grid by exact area overlap (the model raster is never resampled).
Unreadable files are retried, then skipped and counted (see `processing:` in the configuration).

In [ ]:
manifest, records = run_stages(cfg, ["index", "extract"], command="notebook 03")
pd.DataFrame([r.row() for r in records])[["unit_id", "status", "reason_code", "elapsed_s"]]

In [ ]:
out = Path(cfg.output_dir)
name = cfg.raster_extractions[0].name
ext = pd.read_parquet(out / f"{name}_extractions.parquet")
print(ext.groupby("extraction_method")["et_mm"].describe()[["count", "mean", "min", "max"]])
pd.read_csv(out / f"{name}_exclusions.csv")["reason_code"].value_counts()

`fp_weight_retained_frac`, `fp_frac_in_model_extent` and `fp_domain_capture_frac` are different
quantities with different denominators; they are reported side by side, never multiplied.

In [ ]:
ext.query("extraction_method == 'footprint_conservative'")[
    ["date", "model", "et_mm", "fp_weight_retained_frac", "fp_frac_in_model_extent",
     "n_cells_valid"]].head()

## Landscape representativeness

What land each support sees, from a categorical map moved by exact overlap (classes are never
resampled). Fractions describe the landscape; they are not accuracy guarantees.

In [ ]:
manifest, records = run_stages(cfg, ["analyze"], command="notebook 03",
                               select=lambda u: u.analysis == "representativeness")
rid = cfg.representativeness[0].representativeness_id if cfg.representativeness else None
if rid:
    fr = pd.read_parquet(out / "representativeness" / f"{rid}_class_fractions.parquet")
    print(fr.groupby(["extraction_method", "class_name"])["class_frac_of_valid"].mean().round(3))